# 02 — Model Training

**Project:** AuraVision — Crowd Age Group Majority Classifier  
**Purpose:** Train the EfficientNet-B0 transfer-learning model on the split dataset.

All model logic lives in `src/train_model.py`.  
This notebook calls those functions and displays training progress.

**Output:** `models/age_classifier.pth`

## Step 0 — Setup

In [ ]:
!pip install -q torch torchvision pillow numpy

import os, sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AuraVision'
except ImportError:
    PROJECT_ROOT = os.path.abspath('..')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
print('Working directory:', os.getcwd())

## Step 1 — Load Training Data

In [ ]:
from train_model import build_dataloaders, TRAIN_DIR, DEVICE

print(f'Device: {DEVICE}\n')
train_loader, val_loader, class_to_idx = build_dataloaders(TRAIN_DIR)

## Step 2 — Build Model

EfficientNet-B0 pretrained on ImageNet.  
Backbone layers 0–5 are frozen; layers 6+ and the custom classifier head are trainable.

In [ ]:
from train_model import build_model

model = build_model()

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params:     {total:,}')
print(f'Trainable params: {trainable:,}  ({100*trainable/total:.1f}%)')

## Step 3 — Train

Trains for 40 epochs with AdamW + cosine annealing.  
Best checkpoint (by validation accuracy) is saved to `models/age_classifier.pth`.

In [ ]:
from train_model import train_model, NUM_EPOCHS

print(f'Training for {NUM_EPOCHS} epochs...\n')
model = train_model(model, train_loader, val_loader)

## Step 4 — Evaluate on Test Set

In [ ]:
from train_model import build_test_loader, run_inference, TEST_DIR

test_loader, test_dataset = build_test_loader(TEST_DIR, class_to_idx)
print(f'Test images: {len(test_dataset)}\n')
run_inference(model, test_loader, test_dataset)